# 04 Master Database Review

Review the SQLite database generated by Component 4. This notebook checks tables, counts, and example joins for the portfolio database layer.


In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DB_PATH = ROOT / "data/processed/events.db"
REPORT_PATH = ROOT / "reports/master_build_report.json"
DB_PATH


In [ ]:
report = json.loads(REPORT_PATH.read_text())
report


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
tables


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    counts = pd.DataFrame([
        {"table": table, "rows": pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table}", conn).loc[0, "n"]}
        for table in tables["name"]
    ])
counts


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    event_articles = pd.read_sql_query("""
        SELECT e.event_id, e.event_title, e.article_count, l.title, l.media, l.published_at, l.url
        FROM events e
        LEFT JOIN article_event_links l ON e.event_id = l.event_id
        ORDER BY e.event_number, l.title
    """, conn)
event_articles.head(20)


## Review note

The SQLite database is the contract that later API and dashboard components will consume. If this notebook fails, the product layer is not ready.
